# LeetCode #43 — Multiply Strings: nhân tay hai số lớn dạng chuỗi

**Mức độ:** Medium · **Chủ đề:** Math, String, Simulation

Skeleton trên LeetCode:

```python
class Solution:
    def multiply(self, num1: str, num2: str) -> str:
```

Ở bài #2 Add Two Numbers ta mô phỏng phép **cộng** có nhớ. Bài này mô phỏng phép **nhân** như học ở tiểu học, nhưng tổ chức gọn trong **một mảng kết quả**.

Mục tiêu của notebook:

1. Nhìn lại phép nhân đặt tính: tích từng cặp chữ số rơi vào **hàng nào**.
2. Quy tắc vị trí then chốt: chữ số thứ `i` của `num1` nhân chữ số thứ `j` của `num2` đóng góp vào ô **`i + j + 1`** (và nhớ sang ô `i + j`).
3. Vì sao kết quả có **tối đa `m + n` chữ số**.
4. Xử lý số `0` ở đầu kết quả.

# 1. Bài toán là gì?

Cho hai số nguyên **không âm** `num1`, `num2` dạng **chuỗi**. Trả về tích, cũng dạng chuỗi.

**Không được** dùng thư viện số lớn hoặc **chuyển thẳng** input sang số nguyên.

**Constraints (theo đề):**

- `1 <= num1.length, num2.length <= 200`
- `num1`, `num2` chỉ gồm chữ số.
- Không có số `0` ở đầu, trừ khi chính số đó là `"0"`.

---

## Ví dụ

```text
num1 = "2",   num2 = "3"    ->  "6"
num1 = "123", num2 = "456"  ->  "56088"
```

Lưu ý: số 200 chữ số **vượt xa** 64-bit. Trong Python `int` lớn tùy ý nên `str(int(a) * int(b))` chạy được — nhưng đề **cấm**, và trong các ngôn ngữ khác nó không chạy.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Tính toán theo từng chữ số (cộng / nhân có nhớ)
2. Chuỗi (string) trong Python

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Tính toán theo từng chữ số (cộng / nhân có nhớ)

### Cộng tay có nhớ

Ta cộng từ **hàng đơn vị** sang trái; mỗi cột: `tổng = a + b + nhớ`, ghi `tổng % 10`, nhớ `tổng // 10` sang cột kế.

```text
      1 1         ← nhớ
      4 8 7
  +   3 6 5
  ---------
      8 5 2
```

In [1]:
def cong_chuoi(x, y):
    i, j, nho, out = len(x) - 1, len(y) - 1, 0, []
    while i >= 0 or j >= 0 or nho:
        a_ = int(x[i]) if i >= 0 else 0
        b_ = int(y[j]) if j >= 0 else 0
        tong = a_ + b_ + nho
        out.append(str(tong % 10))
        nho = tong // 10
        print(f"   cột: {a_} + {b_} + nhớ → {tong}: ghi {tong % 10}, nhớ {nho}")
        i -= 1; j -= 1
    return "".join(reversed(out))

print("487 + 365 =", cong_chuoi("487", "365"))
print("999 + 1 =", cong_chuoi("999", "1"), "(nhớ ở cột cuối tạo thêm chữ số)")

   cột: 7 + 5 + nhớ → 12: ghi 2, nhớ 1
   cột: 8 + 6 + nhớ → 15: ghi 5, nhớ 1
   cột: 4 + 3 + nhớ → 8: ghi 8, nhớ 0
487 + 365 = 852
   cột: 9 + 1 + nhớ → 10: ghi 0, nhớ 1
   cột: 9 + 0 + nhớ → 10: ghi 0, nhớ 1
   cột: 9 + 0 + nhớ → 10: ghi 0, nhớ 1
   cột: 0 + 0 + nhớ → 1: ghi 1, nhớ 0
999 + 1 = 1000 (nhớ ở cột cuối tạo thêm chữ số)


### Nhân tay: chữ số hàng `i` × chữ số hàng `j` rơi vào hàng `i + j`

Đánh số từ **phải** (hàng đơn vị là 0): tích hai chữ số ở hàng `i` và `j` đóng góp vào hàng `i + j` của kết quả. Cộng dồn mọi đóng góp rồi xử lý nhớ một lượt. Tích của số `m` chữ số và `n` chữ số có tối đa `m + n` chữ số.

```text
        1 2 3          hàng:  2 1 0
      ×   4 5                   1 0
      -------
        6 1 5          123 × 5
      4 9 2            123 × 4, dịch một hàng
      -------
      5 5 3 5
```

In [2]:
def nhan_chuoi(x, y):
    cot = [0] * (len(x) + len(y))           # cot[k] = tổng đóng góp vào hàng k (từ phải)
    for i, dx in enumerate(reversed(x)):
        for j, dy in enumerate(reversed(y)):
            cot[i + j] += int(dx) * int(dy)
    nho = 0
    for k in range(len(cot)):               # xử lý nhớ
        nho, cot[k] = divmod(cot[k] + nho, 10)
    s = "".join(map(str, reversed(cot))).lstrip("0")
    return s or "0"

print("123 × 45 =", nhan_chuoi("123", "45"), "| kiểm tra:", 123 * 45)
print("0 × 999 =", nhan_chuoi("0", "999"))

123 × 45 = 5535 | kiểm tra: 5535
0 × 999 = 0


Lỗi hay gặp: quên **nhớ ở cột cuối** (`999 + 1`); đọc chữ số từ trái trong khi phải cộng từ phải; quên bỏ số 0 đầu (`"00"` → `"0"`); hai số dài khác nhau — hết chữ số thì coi là 0.

## 2.2 Chuỗi (string) trong Python

### Chuỗi là bất biến

`str` **không sửa được** tại chỗ: `s[0] = "x"` gây lỗi. Mọi "sửa" đều tạo chuỗi mới. Vì vậy:

- Ghép nhiều mảnh: gom vào **list** rồi `"".join(list)` (`O(n)`), tránh `s += ...` trong vòng lặp (có thể `O(n²)`).
- Cần sửa từng ký tự: `chars = list(s)`, sửa, rồi `"".join(chars)`.

In [3]:
s_str = "hello"
try:
    s_str[0] = "H"
except TypeError as e:
    print("Sửa trực tiếp:", e)
chars_str = list(s_str); chars_str[0] = "H"
print("qua list:", "".join(chars_str))
manh = []
for i in range(5):
    manh.append(str(i))
print("join:", "-".join(manh))

Sửa trực tiếp: 'str' object does not support item assignment
qua list: Hello
join: 0-1-2-3-4


### Cắt chuỗi (slicing) và các thao tác hay dùng

| Biểu thức | Ý nghĩa |
|---|---|
| `s[i:j]` | ký tự từ `i` tới `j−1` (tạo chuỗi mới, `O(j−i)`) |
| `s[::-1]` | chuỗi đảo ngược |
| `s.split()` | tách theo khoảng trắng (bỏ khoảng trắng thừa) |
| `s.split(",")` | tách theo dấu phẩy (giữ phần rỗng) |
| `s.strip()` | bỏ khoảng trắng hai đầu |
| `s.startswith(p)`, `s.find(p)` | kiểm tra tiền tố, tìm vị trí |
| `c.isdigit()`, `c.isalpha()`, `c.isalnum()`, `c.isspace()` | phân loại ký tự |
| `c.lower()`, `c.upper()` | đổi hoa/thường |

In [4]:
t_str = "  Xin chào   thế giới  "
print(repr(t_str.split()), "|", repr(t_str.strip()), "|", repr("a,,b".split(",")))
print("đảo:", "abcdef"[::-1], "| cắt [1:4]:", "abcdef"[1:4], "| 'abc'.find('c') =", "abc".find("c"))
print([(c, c.isdigit(), c.isalpha()) for c in "a1 "])

['Xin', 'chào', 'thế', 'giới'] | 'Xin chào   thế giới' | ['a', '', 'b']
đảo: fedcba | cắt [1:4]: bcd | 'abc'.find('c') = 2
[('a', False, True), ('1', True, False), (' ', False, False)]


### Ký tự và mã số: `ord` / `chr`

Mỗi ký tự có một mã số (Unicode). `ord("a") = 97`, `chr(97) = "a"`. Mẹo: chữ số `c` → giá trị `ord(c) - ord("0")`; chữ thường `c` → chỉ số 0..25 là `ord(c) - ord("a")`.

In [5]:
print("ord('a') =", ord("a"), "| chr(98) =", chr(98))
print("'7' → 7:", ord("7") - ord("0"), "| 'd' → chỉ số:", ord("d") - ord("a"))
dem26 = [0] * 26
for c in "banana":
    dem26[ord(c) - ord("a")] += 1
print("đếm 26 chữ cái của 'banana' (a, b, n):", dem26[0], dem26[1], dem26[13])

ord('a') = 97 | chr(98) = b
'7' → 7: 7 | 'd' → chỉ số: 3
đếm 26 chữ cái của 'banana' (a, b, n): 3 1 2


### Gom nhóm ký tự liên tiếp

`itertools.groupby` gom các phần tử **liên tiếp bằng nhau** — tiện cho mã hóa độ dài loạt (run-length): `"aaabcc" → 3a 1b 2c`. Tự viết bằng hai con trỏ: `i` ở đầu nhóm, `j` chạy tới khi ký tự khác.

In [6]:
from itertools import groupby
print([(k, len(list(g))) for k, g in groupby("aaabcc")])

def run_length(s):
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        out.append(f"{j - i}{s[i]}")
        i = j
    return "".join(out)
print("run_length('aaabcc') =", run_length("aaabcc"))

[('a', 3), ('b', 1), ('c', 2)]
run_length('aaabcc') = 3a1b2c


Lỗi hay gặp: `split(" ")` sinh phần rỗng khi có nhiều dấu cách; so sánh chuỗi số `"10" < "9"` là **True** (so từ điển, không phải số); nối chuỗi trong vòng lặp lớn; quên chuỗi rỗng.

## 2.3 Áp dụng vào bài này

Không được đổi cả chuỗi thành số → mô phỏng **phép nhân tay**: chữ số hàng `i` nhân chữ số hàng `j` cộng vào ô `i + j` của mảng kết quả dài `m + n`, xử lý nhớ, rồi ghép lại thành chuỗi và bỏ số 0 đầu.

# 3. Nền tảng: đổi ký tự chữ số thành số

`ord(c) - ord('0')` (đã gặp ở bài #8), hoặc `int(c)` cho **một** ký tự (đổi từng chữ số là được phép; cái bị cấm là đổi cả chuỗi lớn).

In [7]:
from typing import List

for c in "0379":
    print(repr(c), "->", ord(c) - ord("0"))

'0' -> 0
'3' -> 3
'7' -> 7
'9' -> 9


# 4. Nhìn lại phép nhân đặt tính

```text
            1 2 3        num1 (m = 3 chữ số)
          x 4 5 6        num2 (n = 3 chữ số)
          -------
            7 3 8        123 x 6
          6 1 5          123 x 5, lùi 1 hàng
        4 9 2            123 x 4, lùi 2 hàng
        ---------
        5 6 0 8 8
```

Mỗi **tích của một cặp chữ số** `num1[i] × num2[j]` rơi vào một **hàng** xác định. Thay vì tạo từng dòng rồi cộng, ta **cộng thẳng mọi tích vào đúng hàng** trong một mảng.

## Tích rơi vào hàng nào?

Đánh chỉ số từ **trái** (như chuỗi). Với `m = len(num1)`, `n = len(num2)`:

- Chữ số `num1[i]` ở hàng (tính từ phải, hàng đơn vị là 0) `m - 1 - i`.
- Chữ số `num2[j]` ở hàng `n - 1 - j`.
- Tích ở hàng `(m - 1 - i) + (n - 1 - j)`.

Nếu mảng kết quả `res` dài `m + n` và cũng đánh chỉ số **từ trái**, hàng `h` (tính từ phải) ứng với ô `m + n - 1 - h`. Thay vào:

```text
ô = m + n - 1 - [(m - 1 - i) + (n - 1 - j)] = i + j + 1
```

> **Quy tắc vị trí:** `num1[i] × num2[j]` cộng vào `res[i + j + 1]`; phần nhớ đẩy sang `res[i + j]`.

In [8]:
num1, num2 = "123", "456"
m, n = len(num1), len(num2)
for i in range(m):
    for j in range(n):
        print(f"num1[{i}]={num1[i]} x num2[{j}]={num2[j]} = {int(num1[i]) * int(num2[j]):>2}  -> ô {i + j + 1}")

num1[0]=1 x num2[0]=4 =  4  -> ô 1
num1[0]=1 x num2[1]=5 =  5  -> ô 2
num1[0]=1 x num2[2]=6 =  6  -> ô 3
num1[1]=2 x num2[0]=4 =  8  -> ô 2
num1[1]=2 x num2[1]=5 = 10  -> ô 3
num1[1]=2 x num2[2]=6 = 12  -> ô 4
num1[2]=3 x num2[0]=4 = 12  -> ô 3
num1[2]=3 x num2[1]=5 = 15  -> ô 4
num1[2]=3 x num2[2]=6 = 18  -> ô 5


## Vì sao `m + n` chữ số là đủ?

`num1 < 10^m` và `num2 < 10^n` nên `num1 × num2 < 10^(m+n)` — tức tích có **tối đa `m + n` chữ số**. (Tối thiểu `m + n - 1` chữ số nếu cả hai khác 0.)

In [9]:
print(len(str(999 * 99)), "<=", 3 + 2)
print(len(str(100 * 10)), ">=", 3 + 2 - 1)

5 <= 5
4 >= 4


# 5. Cách 1: nhân từng chữ số rồi cộng chuỗi

Cách "đúng như trên giấy": với mỗi chữ số của `num2`, nhân cả `num1` với nó, thêm số `0` vào cuối (lùi hàng), rồi cộng dồn bằng hàm cộng chuỗi (giống bài Add Strings).

In [10]:
def add_strings(a: str, b: str) -> str:
    i, j, carry, out = len(a) - 1, len(b) - 1, 0, []
    while i >= 0 or j >= 0 or carry:
        x = ord(a[i]) - 48 if i >= 0 else 0
        y = ord(b[j]) - 48 if j >= 0 else 0
        carry, d = divmod(x + y + carry, 10)
        out.append(chr(d + 48))
        i -= 1
        j -= 1
    return "".join(reversed(out))


def multiply_by_digit(a: str, d: int) -> str:
    if d == 0:
        return "0"
    carry, out = 0, []
    for k in range(len(a) - 1, -1, -1):
        carry, digit = divmod((ord(a[k]) - 48) * d + carry, 10)
        out.append(chr(digit + 48))
    if carry:
        out.append(chr(carry + 48))
    return "".join(reversed(out))


def multiply_rows(num1: str, num2: str) -> str:
    if num1 == "0" or num2 == "0":
        return "0"
    total = "0"
    n = len(num2)
    for j in range(n - 1, -1, -1):
        row = multiply_by_digit(num1, ord(num2[j]) - 48)
        if row != "0":
            row += "0" * (n - 1 - j)          # lùi hàng
        total = add_strings(total, row)
    return total

print(multiply_by_digit("123", 6), multiply_by_digit("123", 5), multiply_by_digit("123", 4))
print(multiply_rows("123", "456"))

738 615 492
56088


Cách này **đúng** và cũng `O(m · n)`, nhưng tạo ra nhiều chuỗi trung gian và cần hai hàm phụ. Cách 2 gọn hơn nhiều.

# 6. Cách 2 (chính): mảng vị trí

```text
res = [0] * (m + n)
for i từ m-1 về 0:
    for j từ n-1 về 0:
        p = digit(num1[i]) * digit(num2[j]) + res[i + j + 1]
        res[i + j + 1] = p % 10
        res[i + j]    += p // 10
bỏ các số 0 ở đầu res, ghép thành chuỗi
```

Duyệt **từ phải sang trái** để số nhớ luôn được đẩy về ô bên trái (chưa xử lý xong), giống cộng tay.

`res[i + j]` có thể tạm thời **lớn hơn 9** — không sao, vì ở vòng sau nó sẽ được cộng vào `p` và tách lại bằng `% 10`, `// 10`.

In [11]:
def multiply(num1: str, num2: str) -> str:
    if num1 == "0" or num2 == "0":
        return "0"
    m, n = len(num1), len(num2)
    res = [0] * (m + n)

    for i in range(m - 1, -1, -1):
        d1 = ord(num1[i]) - ord("0")
        for j in range(n - 1, -1, -1):
            d2 = ord(num2[j]) - ord("0")
            p = d1 * d2 + res[i + j + 1]
            res[i + j + 1] = p % 10
            res[i + j] += p // 10

    # bỏ số 0 ở đầu (tối đa một số 0, vì tích có ít nhất m + n - 1 chữ số)
    start = 0
    while start < len(res) - 1 and res[start] == 0:
        start += 1
    return "".join(str(d) for d in res[start:])

print(multiply("2", "3"))
print(multiply("123", "456"))
print(multiply("0", "52"))
print(multiply("999", "999"))

6
56088
0
998001


# 7. Mô phỏng từng bước: `"123" × "456"`

In [12]:
def trace_multiply(num1: str, num2: str) -> str:
    m, n = len(num1), len(num2)
    res = [0] * (m + n)
    print(f"{num1} x {num2}, res có {m + n} ô: {res}")
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            d1, d2 = int(num1[i]), int(num2[j])
            p = d1 * d2 + res[i + j + 1]
            res[i + j + 1] = p % 10
            res[i + j] += p // 10
            print(f"i={i} j={j}: {d1}x{d2} + (cũ ô {i + j + 1}) = {p:>2} -> ô {i + j + 1} = {p % 10}, ô {i + j} += {p // 10}  res = {res}")
    s = "".join(map(str, res)).lstrip("0") or "0"
    print("Kết quả:", s)
    return s

_ = trace_multiply("123", "456")

123 x 456, res có 6 ô: [0, 0, 0, 0, 0, 0]
i=2 j=2: 3x6 + (cũ ô 5) = 18 -> ô 5 = 8, ô 4 += 1  res = [0, 0, 0, 0, 1, 8]
i=2 j=1: 3x5 + (cũ ô 4) = 16 -> ô 4 = 6, ô 3 += 1  res = [0, 0, 0, 1, 6, 8]
i=2 j=0: 3x4 + (cũ ô 3) = 13 -> ô 3 = 3, ô 2 += 1  res = [0, 0, 1, 3, 6, 8]
i=1 j=2: 2x6 + (cũ ô 4) = 18 -> ô 4 = 8, ô 3 += 1  res = [0, 0, 1, 4, 8, 8]
i=1 j=1: 2x5 + (cũ ô 3) = 14 -> ô 3 = 4, ô 2 += 1  res = [0, 0, 2, 4, 8, 8]
i=1 j=0: 2x4 + (cũ ô 2) = 10 -> ô 2 = 0, ô 1 += 1  res = [0, 1, 0, 4, 8, 8]
i=0 j=2: 1x6 + (cũ ô 3) = 10 -> ô 3 = 0, ô 2 += 1  res = [0, 1, 1, 0, 8, 8]
i=0 j=1: 1x5 + (cũ ô 2) =  6 -> ô 2 = 6, ô 1 += 0  res = [0, 1, 6, 0, 8, 8]
i=0 j=0: 1x4 + (cũ ô 1) =  5 -> ô 1 = 5, ô 0 += 0  res = [0, 5, 6, 0, 8, 8]
Kết quả: 56088


`res` ban đầu `[0, 0, 0, 0, 0, 0]` và kết thúc `[0, 5, 6, 0, 8, 8]` → bỏ số `0` đầu → `"56088"`.

# 8. Phiên bản đúng format LeetCode

In [13]:
class Solution:
    def multiply(self, num1: str, num2: str) -> str:
        if num1 == "0" or num2 == "0":
            return "0"
        m, n = len(num1), len(num2)
        res = [0] * (m + n)
        for i in range(m - 1, -1, -1):
            d1 = ord(num1[i]) - 48
            for j in range(n - 1, -1, -1):
                p = d1 * (ord(num2[j]) - 48) + res[i + j + 1]
                res[i + j + 1] = p % 10
                res[i + j] += p // 10
        start = 1 if res[0] == 0 else 0
        return "".join(map(str, res[start:]))

Vì tích của hai số **khác 0** có `m + n - 1` hoặc `m + n` chữ số, chỉ có thể có **tối đa một** số `0` thừa ở đầu — nên `start = 1 if res[0] == 0 else 0` là đủ.

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<24} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.multiply("2", "3"), "6")
check("Ví dụ 2", sol.multiply("123", "456"), "56088")
check("Nhân với 0", sol.multiply("0", "12345"), "0")
check("0 x 0", sol.multiply("0", "0"), "0")
check("Nhân với 1", sol.multiply("1", "987"), "987")
check("Đủ m + n chữ số", sol.multiply("99", "99"), "9801")
check("Thiếu 1 chữ số", sol.multiply("10", "10"), "100")

Ví dụ 1                  PASS  6
Ví dụ 2                  PASS  56088
Nhân với 0               PASS  0
0 x 0                    PASS  0
Nhân với 1               PASS  987
Đủ m + n chữ số          PASS  9801
Thiếu 1 chữ số           PASS  100


In [15]:
import random

random.seed(43)
def rand_num(max_len):
    k = random.randint(1, max_len)
    if k == 1:
        return str(random.randint(0, 9))
    return str(random.randint(1, 9)) + "".join(random.choice("0123456789") for _ in range(k - 1))

for _ in range(3000):
    a, b = rand_num(40), rand_num(40)
    expected = str(int(a) * int(b))           # chỉ dùng để KIỂM TRA
    assert sol.multiply(a, b) == expected, (a, b)
    assert multiply_rows(a, b) == expected, (a, b)
a, b = "9" * 200, "9" * 200
assert sol.multiply(a, b) == str(int(a) * int(b))
print("3000 test ngẫu nhiên + 200 chữ số x 200 chữ số: đúng")

3000 test ngẫu nhiên + 200 chữ số x 200 chữ số: đúng


# 9. Vì sao đúng? Độ phức tạp?

**Bất biến:** sau mỗi bước `(i, j)`, giá trị số học của mảng `res` (đọc như số thập phân, cho phép ô > 9) bằng tổng các tích `num1[i'] · num2[j'] · 10^(hàng)` đã xử lý. Mỗi bước:

- cộng `d1 · d2` vào ô `i + j + 1` (đúng hàng — mục 4),
- rồi **chuẩn hóa** ô đó (`% 10`) và đẩy phần dư sang ô bên trái (`// 10` tương đương nhân 10).

Thao tác chuẩn hóa không đổi giá trị số học. Ô `res[i + j]` vừa nhận số nhớ sẽ được chuẩn hóa ở một bước **sau đó**, khi nó đóng vai trò `res[i' + j' + 1]` (bước `(i, j - 1)` nếu `j > 0`, hoặc bước `(i - 1, 0)` nếu `j = 0`). Riêng `res[0]` không bao giờ được chuẩn hóa — nhưng vì tích `< 10^(m+n)`, nó chắc chắn `< 10`. Vậy cuối cùng mọi ô đều là một chữ số.

**Độ phức tạp:**

```text
Time:  O(m · n)       — mỗi cặp chữ số một phép nhân
Space: O(m + n)       — mảng kết quả
```

Có thuật toán nhanh hơn cho số rất lớn (Karatsuba `O(n^1.585)`, FFT `O(n log n)`), nhưng với 200 chữ số thì `O(m · n)` là đủ.

# 10. Những lỗi dễ mắc

## Lỗi 1: Đổi thẳng sang `int`

Vi phạm yêu cầu đề. Ngôn ngữ khác sẽ tràn số.

## Lỗi 2: Đặt tích vào ô `i + j` thay vì `i + j + 1`

In [16]:
def wrong_position(num1: str, num2: str) -> str:
    m, n = len(num1), len(num2)
    res = [0] * (m + n)
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            p = int(num1[i]) * int(num2[j]) + res[i + j]
            res[i + j] = p % 10                                 # SAI: lệch một hàng
            if i + j - 1 >= 0:
                res[i + j - 1] += p // 10
    return "".join(map(str, res)).lstrip("0") or "0"

print("Sai :", wrong_position("123", "456"), "| Đúng:", multiply("123", "456"))

Sai : 560880 | Đúng: 56088


Kết quả bị dịch sai **một hàng** (gấp 10 lần). Với input khác, số nhớ ở hàng cao nhất còn có thể bị **mất** vì không còn ô nào bên trái ô `0`.

## Lỗi 3: Không bỏ số `0` ở đầu

`"10" × "10"` → `res = [0, 1, 0, 0]` → phải trả `"100"`, không phải `"0100"`.

## Lỗi 4: Bỏ **hết** số `0` khi kết quả là `0`

`"0".lstrip("0")` là chuỗi rỗng. Nếu bỏ số 0 bằng `lstrip`, phải thêm `or "0"`; hoặc xử lý trường hợp có `"0"` ngay từ đầu như lời giải chính.

In [17]:
print(repr("0000".lstrip("0")), "->", repr("0000".lstrip("0") or "0"))

'' -> '0'


# 11. Mẹo nhận diện pattern

**Số lớn dạng chuỗi / mảng chữ số → mô phỏng phép tính tay:**

| Phép | Bài | Ý chính |
|---|---|---|
| Cộng | #2, #415, #67 | duyệt từ phải, `carry` |
| Nhân | #43 | mảng `m + n`, ô `i + j + 1` |
| Cộng 1 | #66 Plus One | `carry` lan từ cuối |

Cách "**cộng mọi đóng góp vào đúng vị trí trước, chuẩn hóa sau**" cũng là ý tưởng của **phép nhân đa thức** (tích của `x^i` và `x^j` rơi vào `x^(i+j)`).

## Template

```python
res = [0] * (m + n)
for i in reversed(range(m)):
    for j in reversed(range(n)):
        p = a[i] * b[j] + res[i + j + 1]
        res[i + j + 1] = p % 10
        res[i + j] += p // 10
```

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Tính `"25" × "34"` bằng mảng vị trí: ghi `res` sau mỗi bước `(i, j)`.

## Bài 2 — Nhân chuỗi với một chữ số

Viết hàm nhân chuỗi số `a` với một chữ số `d` (0..9), không dùng `int(a)`.

```python
mul_digit("123", 6)  -> "738"
mul_digit("999", 9)  -> "8991"
mul_digit("5", 0)    -> "0"
```

In [18]:
def exercise_2_mul_digit(a: str, d: int) -> str:
    # TODO: duyệt từ phải, carry
    pass


# assert exercise_2_mul_digit("123", 6) == "738"
# assert exercise_2_mul_digit("999", 9) == "8991"
# assert exercise_2_mul_digit("5", 0) == "0"

## Bài 3 — Điền chỗ trống

In [19]:
def exercise_3_multiply(num1: str, num2: str) -> str:
    if num1 == "0" or num2 == "0":
        return "0"
    m, n = len(num1), len(num2)
    res = [0] * (m + n)
    # TODO 1: hai vòng lặp từ phải sang trái
    # TODO 2: cộng tích vào ô i + j + 1, chuẩn hóa, đẩy nhớ sang ô i + j
    # TODO 3: bỏ số 0 đầu, ghép chuỗi
    pass


# assert exercise_3_multiply("123", "456") == "56088"
# assert exercise_3_multiply("10", "10") == "100"
# assert exercise_3_multiply("0", "7") == "0"

## Bài 4 — Biến thể: bình phương số lớn

Viết hàm tính `num * num` (chuỗi) bằng lời giải chính, rồi dùng nó để tính `2^64` bằng cách bình phương liên tiếp bắt đầu từ `"2"`.

```python
square("12") -> "144"
power_of_two_squarings(6) -> "18446744073709551616"     # 2^(2^6) = 2^64
```

In [20]:
def exercise_4_power_of_two_squarings(k: int) -> str:
    # TODO: bắt đầu s = "2", bình phương k lần
    pass


# assert exercise_4_power_of_two_squarings(0) == "2"
# assert exercise_4_power_of_two_squarings(6) == "18446744073709551616"

## Bài 5 — Mini challenge: nhân đa thức

Đa thức được biểu diễn bằng list hệ số, `p[k]` là hệ số của `x^k`. Nhân hai đa thức.

```python
poly_mul([1, 1], [1, 1])    -> [1, 2, 1]        # (1 + x)^2
poly_mul([2, 0, 3], [1, 4]) -> [2, 8, 3, 12]    # (2 + 3x^2)(1 + 4x)
```

Đây chính là phép nhân số **không có bước nhớ** (hệ số có thể > 9).

In [21]:
def exercise_5_poly_mul(p: List[int], q: List[int]) -> List[int]:
    # TODO
    pass


# assert exercise_5_poly_mul([1, 1], [1, 1]) == [1, 2, 1]
# assert exercise_5_poly_mul([2, 0, 3], [1, 4]) == [2, 8, 3, 12]

# 13. Gợi ý

**Bài 1**
- Mức 1: `m = n = 2`, `res` có 4 ô.
- Mức 2: bước đầu `(i=1, j=1)`: `5 × 4 = 20` → ô 3 = 0, ô 2 += 2.

**Bài 2**
- Mức 1: giống `multiply_by_digit` ở mục 5.

**Bài 4**
- Mức 1: `square(s) = multiply(s, s)`.

**Bài 5**
- Mức 1: `res = [0] * (len(p) + len(q) - 1)`.
- Mức 2: `res[i + j] += p[i] * q[j]` (chỉ số theo bậc, nên là `i + j`, không `+1`).

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [22]:
_ = trace_multiply("25", "34")

25 x 34, res có 4 ô: [0, 0, 0, 0]
i=1 j=1: 5x4 + (cũ ô 3) = 20 -> ô 3 = 0, ô 2 += 2  res = [0, 0, 2, 0]
i=1 j=0: 5x3 + (cũ ô 2) = 17 -> ô 2 = 7, ô 1 += 1  res = [0, 1, 7, 0]
i=0 j=1: 2x4 + (cũ ô 2) = 15 -> ô 2 = 5, ô 1 += 1  res = [0, 2, 5, 0]
i=0 j=0: 2x3 + (cũ ô 1) =  8 -> ô 1 = 8, ô 0 += 0  res = [0, 8, 5, 0]
Kết quả: 850


## Đáp án Bài 2

In [23]:
from typing import List

def answer_2_mul_digit(a: str, d: int) -> str:
    if d == 0:
        return "0"
    carry, out = 0, []
    for k in range(len(a) - 1, -1, -1):
        carry, digit = divmod((ord(a[k]) - 48) * d + carry, 10)
        out.append(str(digit))
    if carry:
        out.append(str(carry))
    return "".join(reversed(out))

assert answer_2_mul_digit("123", 6) == "738"
assert answer_2_mul_digit("999", 9) == "8991"
assert answer_2_mul_digit("5", 0) == "0"
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [24]:
def answer_3_multiply(num1: str, num2: str) -> str:
    if num1 == "0" or num2 == "0":
        return "0"
    m, n = len(num1), len(num2)
    res = [0] * (m + n)
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            p = (ord(num1[i]) - 48) * (ord(num2[j]) - 48) + res[i + j + 1]
            res[i + j + 1] = p % 10
            res[i + j] += p // 10
    start = 1 if res[0] == 0 else 0
    return "".join(map(str, res[start:]))

assert answer_3_multiply("123", "456") == "56088"
assert answer_3_multiply("10", "10") == "100"
assert answer_3_multiply("0", "7") == "0"
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [25]:
def answer_4_power_of_two_squarings(k: int) -> str:
    s = "2"
    for _ in range(k):
        s = multiply(s, s)
    return s

assert answer_4_power_of_two_squarings(0) == "2"
assert answer_4_power_of_two_squarings(6) == "18446744073709551616"
print("Bài 4 PASS, 2^128 =", answer_4_power_of_two_squarings(7))

Bài 4 PASS, 2^128 = 340282366920938463463374607431768211456


## Đáp án Bài 5

In [26]:
def answer_5_poly_mul(p: List[int], q: List[int]) -> List[int]:
    res = [0] * (len(p) + len(q) - 1)
    for i, a in enumerate(p):
        for j, b in enumerate(q):
            res[i + j] += a * b
    return res

assert answer_5_poly_mul([1, 1], [1, 1]) == [1, 2, 1]
assert answer_5_poly_mul([2, 0, 3], [1, 4]) == [2, 8, 3, 12]
print("Bài 5 PASS")

Bài 5 PASS


So sánh: ở bài chính, chỉ số đi **từ trái** (chữ số cao nhất trước) nên dùng `i + j + 1`; ở đa thức, chỉ số là **bậc** (thấp nhất trước) nên dùng `i + j`. Cùng một ý tưởng, chỉ khác cách đánh chỉ số.

# 15. Tổng kết

> `num1[i] × num2[j]` rơi vào ô `i + j + 1` của mảng dài `m + n`; số nhớ sang ô `i + j`. Duyệt từ phải sang trái, cuối cùng bỏ tối đa một số `0` ở đầu.

---

## Checklist trước khi nộp

- [ ] Không đổi cả chuỗi sang `int`?
- [ ] Mảng kết quả dài `m + n`?
- [ ] Cộng vào ô `i + j + 1`, nhớ sang `i + j`?
- [ ] Xử lý `"0"` (trả `"0"`, không phải chuỗi rỗng)?
- [ ] Bỏ số `0` thừa ở đầu?